# 01 — EDA Awal → Cleaning Teknis → EDA Setelah Cleaning
## KulitKu · Empat kelas · Google Colab / Drive

**Cakupan:** inventaris ringan, EDA metadata seluruh file, EDA visual pada sampel terstratifikasi, cleaning teknis dan review manual opsional, lalu perbandingan sebelum–sesudah. Notebook berhenti di EDA setelah cleaning: tidak membuat split, fold, ekstraksi fitur, atau training.

**Lebih ringan dari notebook sebelumnya:** tidak menghitung hash file/piksel, tidak mencari duplikat, tidak membuat snapshot metadata tambahan, dan tidak menghitung indikator visual pada semua 30.684 gambar. Full decode tetap dilakukan satu kali per file yang diizinkan untuk memeriksa keterbacaan; membaca ribuan file dari Drive masih memerlukan waktu. Statistik visual berat hanya pada maksimal 200 gambar per kelas.

**Batas makna cleaning:** keluaran adalah kandidat yang lolos cleaning teknis dan keputusan manual yang tersedia, bukan dataset yang sudah terbukti bebas duplikat/augmentasi bocor. Original status tetap unknown dan group_id kosong. Audit duplikat/lineage wajib dikerjakan terpisah sebelum split.

Dataset sumber tidak diubah/dihapus. Tidak menginstal atau mengganti torch/pandas/Pillow. GPU tidak dibutuhkan. Jalankan sel berurutan; semua import tersedia di notebook ini.


## 1. Import paket dan hubungkan Google Drive
Paket utama memakai versi yang sudah ada di Colab. Jika paket hilang, instal paket yang disebut pada error dan jalankan kembali dari atas setelah restart. Jangan menyalin sel dari tengah tanpa menjalankan import dan konfigurasi.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import os, sys, json, shutil, time, warnings, re, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFile
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value)
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(items, **kwargs): return items
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Di luar Colab: gunakan path lokal pada konfigurasi.')

print('Python:',platform.python_version())
print({x:metadata.version(x) for x in ['numpy','pandas','Pillow','matplotlib']})


## 2. Konfigurasi — sesuaikan path dahulu
- `DATA_SOURCE` langsung berisi Eczema, Herpes Zoster, Normal, Ringworm.
- Bila salinan lokal dari setup masih ada, `USE_LOCAL_COPY=True` memakainya tanpa membaca ulang semua metadata Drive. **Pastikan sendiri salinan itu memang versi dataset yang dimaksud.** Tidak ada klaim kecocokan isi tanpa hash. Path aktual dicatat.
- Jika ragu gunakan `USE_LOCAL_COPY=False` untuk membaca Drive langsung, atau buat ulang salinan dari sumber versi tetap.
- Minimum 128px default hanya flag, bukan alasan pembuangan. Untuk menjadikannya aturan cleaning, pilih `MINIMUM_POLICY='exclude'` dan isi alasan sebelum split.
- Run baru dibuat saat sel konfigurasi dijalankan; salin hasil baru tanpa menimpa audit sebelumnya. Tidak ada resume otomatis yang menebak file sama tanpa hash.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/KulitKu')
DATA_SOURCE = DRIVE_ROOT / 'datasets' / 'original'  # GANTI sesuai lokasi Drive yang benar
LOCAL_DATA_ROOT = Path('/content/kulitku-data/raw')
USE_LOCAL_COPY = True
CLASSES = ['Eczema','Herpes Zoster','Normal','Ringworm']
REPORTED_COUNTS = {'Eczema':6715,'Herpes Zoster':8082,'Normal':7758,'Ringworm':8129}
SEED = 42
MAX_PIXELS = 25_000_000
MIN_SIDE = 128
MINIMUM_POLICY = 'flag_only'  # 'flag_only' atau 'exclude'
MINIMUM_POLICY_REASON = ''    # wajib bila exclude
MAX_WORKERS = 2             # gunakan 1 bila RAM/I/O terbatas; tidak memakai GPU
CHECKPOINT_EVERY = 1000
VISUAL_SAMPLE_PER_CLASS = 200
GALLERY_PER_CLASS = 6
MANUAL_REVIEW_CSV = ''       # opsional: path CSV yang sudah diisi dari template review
ALLOWED_EXTENSIONS = {'.jpg','.jpeg','.png','.webp'}
ALLOWED_FORMATS = {'JPEG','PNG','WEBP'}
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = DRIVE_ROOT / 'reports' / '01_eda_cleaning' / RUN_ID
WORK_DIR = Path('/content/kulitku-eda-cleaning') / RUN_ID
FIG_DIR = WORK_DIR / 'figures'

if MINIMUM_POLICY not in {'flag_only','exclude'}:
    raise ValueError('MINIMUM_POLICY tidak dikenal.')
if MINIMUM_POLICY == 'exclude' and not MINIMUM_POLICY_REASON.strip():
    raise ValueError('Tulis MINIMUM_POLICY_REASON sebelum menerapkan eksklusi minimum.')
if not 1 <= MAX_WORKERS <= 4:
    raise ValueError('MAX_WORKERS harus 1 sampai 4 untuk mengendalikan RAM.')
if VISUAL_SAMPLE_PER_CLASS < 1 or GALLERY_PER_CLASS < 1:
    raise ValueError('Jumlah sampel/galeri harus positif.')
DATA_ROOT = LOCAL_DATA_ROOT if USE_LOCAL_COPY and all((LOCAL_DATA_ROOT/c).is_dir() for c in CLASSES) else DATA_SOURCE
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Lokasi data tidak ada: {DATA_ROOT}. Ubah DATA_SOURCE atau siapkan salinan lokal.')
missing = [c for c in CLASSES if not (DATA_ROOT/c).is_dir()]
if missing:
    print('Isi folder:',[p.name for p in DATA_ROOT.iterdir()][:25])
    raise FileNotFoundError(f'Folder kelas tidak ditemukan: {missing}. DATA_ROOT harus folder induknya.')
for p in [RUN_DIR,WORK_DIR,FIG_DIR]:p.mkdir(parents=True,exist_ok=True)
np.random.seed(SEED)
ImageFile.LOAD_TRUNCATED_IMAGES = False
Image.MAX_IMAGE_PIXELS = MAX_PIXELS
warnings.filterwarnings('error',category=Image.DecompressionBombWarning)
print('Data AKTUAL:',DATA_ROOT)
print('Membaca lokal Colab:',not str(DATA_ROOT).startswith('/content/drive/'))
print('Hash/duplikat/lineage: DITUNDA; tidak dijalankan notebook ini.')
print('Run:',RUN_ID,'| Hasil:',RUN_DIR)


## 3. Temukan file dan tampilkan distribusi awal segera
Tidak ada prepass lstat/hash pada seluruh file. Penelusuran direktori tetap memerlukan akses filesystem; indikator jumlah ditemukan ditampilkan berkala. Semua file dalam empat kelas dicatat, termasuk sidecar dan ekstensi tak didukung. Symlink tidak diikuti.


In [ ]:
def discover_files(root):
    paths=[]
    for label in CLASSES:
        print('Membaca daftar:',label,flush=True)
        for folder,dirs,files in os.walk(root/label,followlinks=False):
            dirs[:] = sorted(d for d in dirs if not (Path(folder)/d).is_symlink())
            for name in sorted(files):
                paths.append(Path(folder)/name)
                if len(paths)%5000 == 0:print('Ditemukan:',len(paths),'file',flush=True)
    return sorted(paths,key=lambda p:p.relative_to(root).as_posix())

all_files=discover_files(DATA_ROOT)
if not all_files:raise RuntimeError('Tidak ada file ditemukan.')
file_index=pd.DataFrame([{'image_id':f'img_{i:07d}',
    'relative_path':p.relative_to(DATA_ROOT).as_posix(),
    'label':p.relative_to(DATA_ROOT).parts[0],'filename':p.name,
    'extension':p.suffix.lower()} for i,p in enumerate(all_files,1)])
# ID berdasarkan urutan path pada run ini, bukan hash dan bukan ID pasien.
# Jangan gabungkan ID dari run berbeda tanpa mengecek relative_path.
initial_counts=file_index.groupby('label').size().reindex(CLASSES,fill_value=0)
display(initial_counts.rename('semua_file_ditemukan').to_frame())
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(CLASSES,initial_counts.values,color=['#3b82f6','#14b8a6','#8b5cf6','#f59e0b'])
ax.set(ylabel='Jumlah file',title='EDA awal: jumlah file sebelum pemeriksaan teknis')
ax.tick_params(axis='x',rotation=15)
plt.tight_layout();plt.savefig(FIG_DIR/'00_counts_before_scan.png',dpi=140);plt.show();plt.close()
file_index.to_csv(WORK_DIR/'file_index.csv',index=False)
unexpected=[{'name':p.name,'kind':'directory' if p.is_dir() else 'file'} for p in DATA_ROOT.iterdir() if p.name not in CLASSES]
pd.DataFrame(unexpected,columns=['name','kind']).to_csv(WORK_DIR/'outside_target_folders.csv',index=False)
print('Total:',len(all_files),'| Item tingkat atas di luar kelas:',len(unexpected))


## 4. Pemeriksaan teknis satu kali per file
Mencatat dimensi, format isi, mode, ukuran file, frame, alpha, orientasi EXIF, dan keberhasilan full decode serta konversi RGB. Tidak ada sha256, pixel hash, dHash, verify-read kedua, atau indikator visual untuk seluruh dataset.

Batas 25MP diberlakukan sebelum decode. Pillow dapat menolak gambar besar saat membuka header, sehingga dimensi file itu mungkin belum terisi. Gambar multiframe dikarantina untuk review karena tujuan penelitian citra statis; hanya frame pertama diperiksa. Tidak mengubah file sumber dan tidak menyimpan metadata EXIF sensitif.


In [ ]:
def inspect_basic(item):
    index_row,path=item
    r={**index_row,'width':None,'height':None,'display_width':None,'display_height':None,
       'format':'','mode':'','n_frames':None,'file_size_bytes':None,
       'megapixels':None,'aspect_ratio':None,'readability_status':'not_checked',
       'readable':None,'rgb_convertible':False,'has_alpha':False,'has_exif':False,
       'exif_orientation':None,'below_minimum':False,'over_maximum':False,
       'extension_format_mismatch':False,'filename_aug_hint':False,
       'sha256':'','hash_status':'deferred','original_status':'unknown',
       'original_status_evidence':'','original_id':'','group_id':'','group_evidence':'',
       'include':None,'audit_note':'','technical_note':'','error_type':'','error_message':''}
    # Regex is only a descriptive filename hint; it never sets origin/group.
    r['filename_aug_hint']=bool(re.search(r'(?:^|[\W_])(aug(?:mented|mentation)?|flip|rot(?:ate)?|zoom|shear)(?=[\W_\d]|$)',path.stem.lower()))
    if path.is_symlink():
        r.update(readability_status='skipped_symlink',technical_note='Symlink tidak dibaca.')
        return r
    try:r['file_size_bytes']=path.stat().st_size
    except OSError as exc:
        r.update(readability_status='io_error',error_type=type(exc).__name__,error_message=str(exc)[:200])
        return r
    if r['extension'] not in ALLOWED_EXTENSIONS:
        r.update(readability_status='unsupported_extension',technical_note='Ekstensi di luar protokol; tidak didekode.')
        return r
    try:
        with Image.open(path) as im:
            w,h=im.size
            r.update(width=w,height=h,display_width=w,display_height=h,format=im.format or '',
                mode=im.mode,n_frames=getattr(im,'n_frames',1),megapixels=w*h/1e6,
                aspect_ratio=w/h if h else None,has_exif=bool(im.info.get('exif')),
                has_alpha=im.mode in ('RGBA','LA') or 'transparency' in im.info,
                below_minimum=w<MIN_SIDE or h<MIN_SIDE,over_maximum=w*h>MAX_PIXELS)
            expected={'.jpg':'JPEG','.jpeg':'JPEG','.png':'PNG','.webp':'WEBP'}[r['extension']]
            r['extension_format_mismatch']=r['format']!=expected
            if r['over_maximum']:
                r.update(readability_status='skipped_pixel_limit',technical_note='Di atas 25MP; tidak didekode.')
                return r
            if r['format'] not in ALLOWED_FORMATS:
                r.update(readability_status='unsupported_content_format',technical_note='Format isi di luar protokol.')
                return r
            im.seek(0);im.load()
            # Conversion confirms RGB compatibility; it does not rewrite the source.
            im.convert('RGB')
            r.update(readability_status='readable',readable=True,rgb_convertible=True)
            try:
                orientation=im.getexif().get(274,1)
                r['exif_orientation']=int(orientation)
                if orientation in (5,6,7,8):r['display_width'],r['display_height']=h,w
            except Exception:
                r['technical_note']='Metadata orientasi tidak dapat dibaca; tinjau.'
    except (Image.DecompressionBombError,Image.DecompressionBombWarning) as exc:
        r.update(readability_status='skipped_safety_limit',over_maximum=True,
                 technical_note='Ditolak batas keselamatan, bukan bukti korup.',error_type=type(exc).__name__)
    except Exception as exc:
        r.update(readability_status='decode_error',readable=False,
                 error_type=type(exc).__name__,error_message=str(exc)[:200])
    return r

def atomic_copy(src,dst):
    dst.parent.mkdir(parents=True,exist_ok=True)
    temp=dst.with_name(dst.name+'.partial')
    shutil.copy2(src,temp)
    if temp.stat().st_size!=src.stat().st_size:raise IOError('Ukuran backup berbeda.')
    os.replace(temp,dst)

def save_table(frame,name):
    frame.to_csv(WORK_DIR/name,index=False)
    atomic_copy(WORK_DIR/name,RUN_DIR/name)
    return frame

def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR/name,dpi=140,bbox_inches='tight')
    plt.show();plt.close()


## 5. Jalankan scan ringan
Progress bar dimulai langsung saat pemeriksaan. Dua worker membaca file secara bersamaan dengan memori dibatasi. Jika Drive lambat, gunakan salinan lokal yang benar; T4 tidak mempercepat akses file. Full decode 30 ribu gambar tetap perlu waktu, tetapi tidak ada pembacaan hash atau statistik berat seluruh gambar.

Checkpoint JSONL disalin tiap 1.000 file. Ini cadangan hasil parsial, bukan resume otomatis; rerun sel ini memeriksa ulang semua file agar tidak menganggap data yang berubah tetap sama.


In [ ]:
started=time.perf_counter()
checkpoint=WORK_DIR/'basic_scan_checkpoint.jsonl'
items=zip(file_index.to_dict('records'),all_files)
records=[]
with checkpoint.open('w',encoding='utf-8') as stream, ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results=executor.map(inspect_basic,items)
    for i,row in enumerate(tqdm(results,total=len(all_files),desc='Baca gambar (tanpa hash)'),1):
        records.append(row)
        stream.write(json.dumps(row,ensure_ascii=False)+'\n')
        if i%CHECKPOINT_EVERY==0:
            stream.flush();atomic_copy(checkpoint,RUN_DIR/checkpoint.name)
    stream.flush()
atomic_copy(checkpoint,RUN_DIR/checkpoint.name)
before=pd.DataFrame(records)
for col in ['width','height','display_width','display_height','n_frames','file_size_bytes','exif_orientation']:
    before[col]=pd.array(before[col],dtype='Int64')
before['readable']=pd.array(before.readable,dtype='boolean')
before['include']=pd.array(before['include'],dtype='Int64')
assert len(before)==len(file_index) and before.image_id.is_unique
save_table(before,'manifest_before_cleaning.csv')
scan_seconds=time.perf_counter()-started
print('Selesai:',len(before),'file | menit:',round(scan_seconds/60,2))
print(before.readability_status.value_counts().to_string())


## 6. EDA awal lengkap — kelas, format, mode, keterbacaan dan ukuran
Semua file masuk hitungan awal; hanya gambar yang memiliki metadata valid masuk grafik dimensi. Statistik kualitas visual dan contoh gambar akan disajikan pada bagian sampel. Label adalah label folder, belum diagnosis yang diverifikasi.


In [ ]:
def numeric_summary(frame,cols):
    rows=[]
    for label in CLASSES:
        for col in cols:
            v=pd.to_numeric(frame.loc[frame.label.eq(label),col],errors='coerce').dropna().astype(float)
            rows.append({'label':label,'metric':col,'n':len(v),
                'mean':float(v.mean()) if len(v) else None,'std':float(v.std(ddof=1)) if len(v)>1 else None,
                'min':float(v.min()) if len(v) else None,'p05':float(v.quantile(.05)) if len(v) else None,
                'median':float(v.median()) if len(v) else None,'p95':float(v.quantile(.95)) if len(v) else None,
                'max':float(v.max()) if len(v) else None})
    return pd.DataFrame(rows)


def metadata_eda(frame,stage):
    summary=pd.DataFrame({'label':CLASSES})
    summary['reported']=summary.label.map(REPORTED_COUNTS)
    summary['all_files']=summary.label.map(frame.label.value_counts()).fillna(0).astype(int)
    summary['image_extension_files']=summary.label.map(frame.loc[frame.extension.isin(ALLOWED_EXTENSIONS),'label'].value_counts()).fillna(0).astype(int)
    summary['readable']=summary.label.map(frame.loc[frame.readability_status.eq('readable'),'label'].value_counts()).fillna(0).astype(int)
    summary['percent_all_files']=summary.all_files/max(len(frame),1)*100
    summary['delta_image_ext_vs_reported']=summary.image_extension_files-summary.reported
    save_table(summary,f'{stage}_class_summary.csv');display(summary)
    fig,ax=plt.subplots(figsize=(10,4))
    summary.set_index('label')[['all_files','image_extension_files','readable']].plot.bar(ax=ax,rot=15)
    ax.set(ylabel='Jumlah',title=f'{stage}: kelas dan keterbacaan');save_figure(f'{stage}_01_classes.png')
    statuses=pd.crosstab(frame.label,frame.readability_status).reindex(CLASSES,fill_value=0)
    save_table(statuses.reset_index(),f'{stage}_readability.csv');display(statuses)
    formats=frame.groupby(['label','extension','format','mode'],dropna=False).size().rename('count').reset_index()
    save_table(formats,f'{stage}_formats_modes.csv');display(formats.head(25))
    numeric=numeric_summary(frame,['width','height','megapixels','aspect_ratio','file_size_bytes'])
    save_table(numeric,f'{stage}_numeric_summary.csv')
    available=frame.dropna(subset=['width','height'])
    if len(available):
        fig,axes=plt.subplots(2,2,figsize=(12,8))
        mp=pd.to_numeric(available.megapixels,errors='coerce').dropna().astype(float)
        ratio=pd.to_numeric(available.aspect_ratio,errors='coerce').dropna().astype(float)
        axes[0,0].hist(mp,bins=35,color='#3b82f6');axes[0,0].set(title='Resolusi',xlabel='Megapiksel',ylabel='Jumlah')
        axes[0,1].hist(ratio,bins=35,color='#14b8a6');axes[0,1].set(title='Rasio aspek',xlabel='Lebar / tinggi',ylabel='Jumlah')
        sampled=available.sample(n=min(5000,len(available)),random_state=SEED)
        for label in CLASSES:
            s=sampled[sampled.label.eq(label)]
            axes[1,0].scatter(s.width.astype(float),s.height.astype(float),s=7,alpha=.3,label=label)
        axes[1,0].axvline(MIN_SIDE,color='black',ls='--');axes[1,0].axhline(MIN_SIDE,color='black',ls='--')
        axes[1,0].set(xscale='log',yscale='log',xlabel='Lebar (log)',ylabel='Tinggi (log)',title='Dimensi; maksimal 5.000 contoh')
        axes[1,0].legend(fontsize=7)
        size=pd.to_numeric(frame.file_size_bytes,errors='coerce').dropna().astype(float)/1024
        axes[1,1].hist(size,bins=35,color='#8b5cf6');axes[1,1].set(title='Ukuran file',xlabel='KiB',ylabel='Jumlah')
        save_figure(f'{stage}_02_dimensions.png')
    modes=pd.crosstab(frame.label,frame['mode'].replace('','unknown')).reindex(CLASSES,fill_value=0)
    if len(modes.columns):
        fig,ax=plt.subplots(figsize=(10,4));modes.plot.bar(ax=ax,stacked=True,rot=15)
        ax.set(title=f'{stage}: mode warna sumber',ylabel='Jumlah');save_figure(f'{stage}_03_modes.png')
    flags=frame.groupby('label').agg(below_minimum=('below_minimum','sum'),over_maximum=('over_maximum','sum'),
        alpha=('has_alpha','sum'),extension_mismatch=('extension_format_mismatch','sum'),filename_aug_hint=('filename_aug_hint','sum')).reindex(CLASSES,fill_value=0)
    save_table(flags.reset_index(),f'{stage}_flags.csv');display(flags)
    directory=frame.assign(parent=frame.relative_path.map(lambda x:Path(x).parent.as_posix())).groupby(['label','parent']).size().rename('count').reset_index()
    save_table(directory,f'{stage}_directory_counts.csv')
    prefixes=frame.assign(prefix=frame.filename.map(lambda x:re.split(r'[_\-\s]',Path(x).stem)[0][:40])).groupby(['label','prefix']).size().rename('count').reset_index().sort_values('count',ascending=False)
    save_table(prefixes,f'{stage}_filename_prefixes.csv')
    return summary

before_summary=metadata_eda(before,'before')
print('Token aug di nama file hanya petunjuk, tidak digunakan mengubah original_status/group_id.')


## 7. Sampel visual terstratifikasi dan indikator kualitas
Ambil maksimal 200 gambar terbaca per kelas dengan seed 42. Sampel ini digunakan sebagai acuan yang sama: EDA visual sesudah cleaning memakai subset dari sampel awal yang dipertahankan, bukan menghitung ulang semua gambar. Jumlah sampel per kelas dilaporkan, karena dapat berkurang setelah cleaning.

RGB brightness/kontras/saturasi, histogram RGB, entropy dan Laplacian dihitung pada thumbnail maksimum 256px. Ini **bukan** deteksi penyakit, skor kualitas klinis, atau pengukuran warna kulit pasien. Alpha dikomposit ke putih untuk statistik; sumber tetap utuh. Laplacian rendah hanya kandidat review, bukan aturan eksklusi.


In [ ]:
def load_rgb(path):
    with Image.open(path) as im:
        if im.width*im.height>MAX_PIXELS:raise ValueError('Di atas batas piksel.')
        im.seek(0);im.load()
        image=ImageOps.exif_transpose(im)
        if image.mode in ('RGBA','LA') or 'transparency' in image.info:
            rgba=image.convert('RGBA');background=Image.new('RGBA',rgba.size,(255,255,255,255))
            return Image.alpha_composite(background,rgba).convert('RGB')
        return image.convert('RGB')

def image_metrics(row):
    r={'image_id':row.image_id,'relative_path':row.relative_path,'label':row.label,
       'metrics_status':'ok','metrics_error':''}
    try:
        image=load_rgb(DATA_ROOT/row.relative_path)
        image.thumbnail((256,256),Image.Resampling.LANCZOS)
        gray=np.asarray(image.convert('L'),dtype=np.float32)
        hsv=np.asarray(image.convert('HSV'),dtype=np.float32)
        arr=np.asarray(image,dtype=np.float32)
        counts=np.bincount(gray.astype('uint8').ravel(),minlength=256).astype(float)
        p=counts[counts>0]/counts.sum()
        lapvar=None
        if min(gray.shape)>=3:
            lap=-4*gray[1:-1,1:-1]+gray[:-2,1:-1]+gray[2:,1:-1]+gray[1:-1,:-2]+gray[1:-1,2:]
            lapvar=float(np.var(lap))
        r.update(brightness_mean=float(gray.mean()),contrast_std=float(gray.std()),
            saturation_mean=float(hsv[:,:,1].mean()/255),entropy=float(-(p*np.log2(p)).sum()),
            laplacian_variance=lapvar,near_black_fraction=float((gray<=10).mean()),
            near_white_fraction=float((gray>=245).mean()),
            mean_r=float(arr[:,:,0].mean()),mean_g=float(arr[:,:,1].mean()),mean_b=float(arr[:,:,2].mean()))
        for channel in range(3):
            hist=np.histogram(arr[:,:,channel],bins=32,range=(0,256))[0].astype(float)
            hist/=hist.sum()
            for bin_id,value in enumerate(hist):r[f'hist_{channel}_{bin_id:02d}']=float(value)
    except Exception as exc:
        r.update(metrics_status='error',metrics_error=type(exc).__name__+': '+str(exc)[:150])
    return r

samples=[]
for label in CLASSES:
    available=before[before.label.eq(label)&before.readability_status.eq('readable')&before.n_frames.eq(1).fillna(False)]
    samples.append(available.sample(n=min(VISUAL_SAMPLE_PER_CLASS,len(available)),random_state=SEED))
visual_index=pd.concat(samples,ignore_index=True)
metric_rows=[image_metrics(row) for row in tqdm(visual_index.itertuples(index=False),total=len(visual_index),desc='Statistik pada SAMPEL')]
metric_cols=['image_id','relative_path','label','metrics_status','metrics_error','brightness_mean','contrast_std',
    'saturation_mean','entropy','laplacian_variance','near_black_fraction','near_white_fraction','mean_r','mean_g','mean_b']
metrics=pd.DataFrame(metric_rows).reindex(columns=metric_cols+[f'hist_{c}_{b:02d}' for c in range(3) for b in range(32)])
save_table(visual_index[['image_id','relative_path','label']],'visual_sample_index.csv')
save_table(metrics,'visual_sample_metrics.csv')
display(visual_index.groupby('label').size().reindex(CLASSES,fill_value=0).rename('sample_n').to_frame())
QUALITY_COLS=['brightness_mean','contrast_std','saturation_mean','entropy','laplacian_variance','near_white_fraction']


def visual_eda(metric_frame,stage):
    good=metric_frame[metric_frame.metrics_status.eq('ok')].copy()
    counts=good.label.value_counts().reindex(CLASSES,fill_value=0)
    save_table(counts.rename('sample_n').rename_axis('label').reset_index(),f'{stage}_visual_sample_counts.csv')
    save_table(numeric_summary(good,QUALITY_COLS),f'{stage}_visual_summary.csv')
    if good.empty:
        print('Tidak ada sampel visual yang dapat dianalisis:',stage);return
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,col in zip(axes.ravel(),QUALITY_COLS):
        arrays=[pd.to_numeric(good.loc[good.label.eq(c),col],errors='coerce').dropna().astype(float).values for c in CLASSES]
        ax.boxplot(arrays,labels=CLASSES,showfliers=False)
        ax.tick_params(axis='x',rotation=25,labelsize=8)
        ax.set_title(col,fontsize=10)
        if col=='laplacian_variance':ax.set_yscale('symlog',linthresh=1)
    fig.suptitle(f'{stage}: indikator visual pada sampel, bukan seluruh dataset')
    save_figure(f'{stage}_04_quality_sample.png')
    fig,axes=plt.subplots(2,2,figsize=(12,7))
    x=np.arange(32)*8+4
    for ax,label in zip(axes.ravel(),CLASSES):
        group=good[good.label.eq(label)]
        for c,color in enumerate(['red','green','blue']):
            hist=group[[f'hist_{c}_{b:02d}' for b in range(32)]].apply(pd.to_numeric,errors='coerce').mean(axis=0).to_numpy(dtype=float)
            if len(group):ax.plot(x,hist,color=color,label=color)
        ax.set(title=f'{label} (n={len(group)})',xlabel='Intensitas RGB',ylabel='Rata-rata proporsi pixel')
        if len(group):ax.legend(fontsize=8)
    save_figure(f'{stage}_05_rgb_histograms_sample.png')
    corr=good[QUALITY_COLS].apply(pd.to_numeric,errors='coerce').corr()
    fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(corr.to_numpy(dtype=float),vmin=-1,vmax=1,cmap='coolwarm')
    ax.set_xticks(range(len(QUALITY_COLS)),QUALITY_COLS,rotation=50,ha='right',fontsize=8)
    ax.set_yticks(range(len(QUALITY_COLS)),QUALITY_COLS,fontsize=8)
    fig.colorbar(im,ax=ax,label='Korelasi Pearson (sampel)');ax.set_title(f'{stage}: korelasi indikator visual')
    save_figure(f'{stage}_06_quality_correlation_sample.png')

visual_eda(metrics,'before')


## 8. Galeri awal, resolusi kecil dan contoh visual ekstrem
Galeri untuk meninjau variasi pencahayaan, latar, area kulit, watermark/teks, label, dan kemiripan. Tidak ada deteksi otomatis validitas diagnosis, watermark, atau kesesuaian area kulit. Penilaian itu dilakukan melalui review manual.


In [ ]:
def gallery(rows,title,filename,columns=3):
    rows=rows.reset_index(drop=True)
    if rows.empty:
        print('Tidak ada contoh:',title);return
    nrows=(len(rows)+columns-1)//columns
    fig,axes=plt.subplots(nrows,columns,figsize=(columns*3.5,nrows*3.4),squeeze=False)
    for ax in axes.ravel():ax.axis('off')
    for ax,row in zip(axes.ravel(),rows.itertuples(index=False)):
        try:
            image=load_rgb(DATA_ROOT/row.relative_path);image.thumbnail((400,400),Image.Resampling.LANCZOS)
            ax.imshow(image);ax.set_title(f'{row.label} | {row.image_id}\n{Path(row.relative_path).name[:34]}',fontsize=8)
        except Exception as exc:ax.text(.05,.5,type(exc).__name__,fontsize=8)
    fig.suptitle(title,fontsize=11);save_figure(filename)

for i,label in enumerate(CLASSES):
    candidates=before[before.label.eq(label)&before.readability_status.eq('readable')]
    sample=candidates.sample(n=min(GALLERY_PER_CLASS,len(candidates)),random_state=SEED)
    gallery(sample,f'Sebelum cleaning: {label}',f'before_gallery_{i}.png')
small=before[before.below_minimum&before.readability_status.eq('readable')].head(6)
gallery(small,'Resolusi kecil: flag review, bukan otomatis tidak layak','review_small_resolution.png')
quality_candidates=[]
for col,ascending,reason in [('brightness_mean',True,'dark_sample'),('brightness_mean',False,'bright_sample'),
                            ('laplacian_variance',True,'low_laplacian_sample'),('entropy',True,'low_entropy_sample')]:
    selected=metrics[metrics.metrics_status.eq('ok')].dropna(subset=[col]).sort_values([col,'image_id'],ascending=[ascending,True]).head(6).copy()
    selected['candidate_reason']=reason;quality_candidates.append(selected)
    gallery(selected,f'Contoh untuk review: {reason} (hanya sampel)',f'review_{reason}.png')
save_table(pd.concat(quality_candidates,ignore_index=True),'quality_sample_review_candidates.csv')


## 9. Cleaning teknis dan review manual opsional
Aturan otomatis:
1. Eksklusi dari kandidat EDA sesudah cleaning: file yang tidak lolos decode/format/batas keselamatan atau bukan file nyata; multiframe dikarantina.
2. Minimum ukuran hanya flag, kecuali kebijakan exclude dipilih dengan alasan di konfigurasi.
3. RGB-convertible grayscale/CMYK/palette/alpha tetap dipertahankan; tidak perlu menyimpan ulang semua gambar. Mismatch ekstensi/format yang isinya tetap diizinkan hanya flag.
4. Tidak membuang berdasarkan blur/brightness/entropy, token augmentasi, atau skor model.

Review manual bersifat opsional: template dihasilkan pada sel ini. Untuk mengeluarkan gambar tidak relevan atau konflik label yang sudah di-review, edit salinan template dan isi keputusan `exclude`, alasan serta reviewer. Keputusan `keep` harus disertai alasan dan tidak boleh melewati kegagalan teknis. Label override membutuhkan bukti sumber/review dan harus salah satu empat kelas.

Setelah mengedit: isi MANUAL_REVIEW_CSV pada konfigurasi, atau tetapkan path di sel baru, lalu jalankan ulang bagian 9 sampai akhir. Tidak perlu mengulang scan jika runtime/variabel masih tersedia. Untuk audit lintas run, template menyertakan relative_path dan harus cocok dengan ID; ID numerik bisa berubah bila daftar file berubah.


In [ ]:
audit=before.copy()
audit['source_label']=audit.label
base_ok=audit.readability_status.eq('readable')&audit.rgb_convertible
audit['technical_keep']=base_ok&audit.n_frames.eq(1).fillna(False)
audit['cleaning_reason']=''
audit.loc[~base_ok,'cleaning_reason']='technical_'+audit.loc[~base_ok,'readability_status']
audit.loc[base_ok&~audit.n_frames.eq(1).fillna(False),'cleaning_reason']='quarantine_multiframe_requires_review'
if MINIMUM_POLICY=='exclude':
    audit.loc[audit.technical_keep&audit.below_minimum,'cleaning_reason']='below_minimum_policy: '+MINIMUM_POLICY_REASON
    audit.loc[audit.below_minimum,'technical_keep']=False
audit['manual_decision']='';audit['manual_reason']='';audit['reviewer']='';audit['label_evidence']=''
audit['retained_for_eda']=audit.technical_keep

review_columns=['image_id','relative_path','label','readability_status','below_minimum','filename_aug_hint']
template=before[review_columns].copy()
for c in ['decision','reason','reviewer','label_override','label_evidence']:template[c]=''
save_table(template,'manual_review_template.csv')
if MANUAL_REVIEW_CSV:
    review=pd.read_csv(MANUAL_REVIEW_CSV,dtype=str,keep_default_na=False)
    required={'image_id','relative_path','decision','reason','reviewer','label_override','label_evidence'}
    if not required.issubset(review.columns):raise ValueError(f'Kolom review kurang: {required-set(review.columns)}')
    review=review[review.decision.str.strip().ne('')|review.label_override.str.strip().ne('')].copy()
    if review.image_id.duplicated().any():raise ValueError('image_id review tidak boleh duplikat.')
    lookup=audit.set_index('image_id')
    if not set(review.image_id).issubset(set(lookup.index)):raise ValueError('Ada image_id review di luar run ini; cek relative_path dan gunakan template yang cocok.')
    for record in review.itertuples(index=False):
        if lookup.loc[record.image_id,'relative_path']!=record.relative_path:raise ValueError('image_id dan relative_path tidak cocok; hentikan agar review salah file tidak diterapkan.')
        decision=record.decision.strip().lower()
        if decision not in {'keep','exclude'}:raise ValueError('Review aktif harus decision keep atau exclude.')
        if not record.reason.strip() or not record.reviewer.strip():raise ValueError('Review memerlukan reason dan reviewer.')
        mask=audit.image_id.eq(record.image_id)
        idx=audit.index[mask][0]
        if decision=='keep' and not bool(audit.at[idx,'technical_keep']):
            raise ValueError('keep tidak boleh melewati kegagalan teknis. Perbaiki kebijakan/sumber lalu scan ulang secara terdokumentasi.')
        proposed=record.label_override.strip()
        if proposed:
            if proposed not in CLASSES or not record.label_evidence.strip():raise ValueError('Label override perlu label target valid dan label_evidence.')
            audit.at[idx,'label']=proposed
        audit.loc[mask,['manual_decision','manual_reason','reviewer','label_evidence']]=[decision,record.reason,record.reviewer,record.label_evidence]
        if decision=='exclude':
            audit.loc[mask,'retained_for_eda']=False
            audit.loc[mask,'cleaning_reason']='manual_exclude: '+record.reason

audit['include']=pd.array([None]*len(audit),dtype='Int64')
audit.loc[~audit.retained_for_eda,'include']=0
audit.loc[audit.retained_for_eda&audit.manual_decision.eq('keep'),'include']=1
audit['audit_note']=audit.apply(lambda r:'; '.join(x for x in [str(r.technical_note),str(r.cleaning_reason),
    'Di bawah minimum sementara; flag saja.' if r.below_minimum and MINIMUM_POLICY=='flag_only' else '',
    'Petunjuk nama augmentasi; belum bukti.' if r.filename_aug_hint else '',
    r.manual_reason,'Audit duplikat/original/lineage belum dilakukan.'] if x),axis=1)
audit['ready_for_split']=False
cleaned=audit[audit.retained_for_eda].copy()
excluded=audit[~audit.retained_for_eda].copy()
save_table(audit,'manifest_cleaning_decisions.csv')
save_table(cleaned,'manifest_after_technical_cleaning.csv')
save_table(excluded,'excluded_or_quarantined.csv')
manual_log=audit[audit.manual_decision.ne('')][['image_id','relative_path','source_label','label','manual_decision','manual_reason','reviewer','label_evidence']]
save_table(manual_log,'manual_decision_log.csv')
assert len(cleaned)+len(excluded)==len(before)
assert cleaned.original_status.eq('unknown').all() and cleaned.group_id.eq('').all()
print('Sebelum:',len(before),'| Dipertahankan untuk EDA:',len(cleaned),'| Dikeluarkan/karantina:',len(excluded))
print('include kosong = menunggu audit; technical_keep/retained_for_eda bukan persetujuan siap split.')
display(excluded[['relative_path','label','readability_status','cleaning_reason']].head(25))


## 10. EDA sesudah cleaning — seluruh metadata dan galeri
Menganalisis kandidat teknis yang dipertahankan, dengan label hasil review jika ada. Tidak ada balancing, augmentasi baru, resize permanen, atau penghapusan gambar sumber.


In [ ]:
after_summary=metadata_eda(cleaned,'after')
for i,label in enumerate(CLASSES):
    subset=cleaned[cleaned.label.eq(label)]
    chosen=subset.sample(n=min(GALLERY_PER_CLASS,len(subset)),random_state=SEED)
    gallery(chosen,f'Setelah cleaning teknis: {label}',f'after_gallery_{i}.png')
excluded_viewable=excluded[excluded.readability_status.eq('readable')].head(6)
gallery(excluded_viewable,'Contoh dikeluarkan/karantina yang masih dapat ditampilkan','excluded_viewable_gallery.png')
# Drop old sample label, attach current audited label by file identity.
after_metrics=(metrics.iloc[0:0].copy() if metrics.empty else
    metrics.drop(columns=['label']).merge(cleaned[['image_id','label']],on='image_id',how='inner',validate='one_to_one'))
save_table(after_metrics,'after_visual_sample_metrics.csv')
visual_eda(after_metrics,'after')
print('Visual sesudah = subset sampel awal yang dipertahankan. Ini bukan pemindaian seluruh gambar.')


## 11. Perbandingan sebelum–sesudah dan dampak cleaning
Hitungan utama mencakup seluruh file awal versus kandidat teknis akhir. Grafik dimensi dibatasi file yang memiliki metadata; grafik visual dibatasi sampel yang dilaporkan. Perubahan label ditampilkan terpisah dari eksklusi supaya tidak disalahartikan sebagai data hilang.


In [ ]:
comparison=pd.DataFrame({'label':CLASSES})
comparison['before_files']=comparison.label.map(before.label.value_counts()).fillna(0).astype(int)
comparison['after_retained']=comparison.label.map(cleaned.label.value_counts()).fillna(0).astype(int)
comparison['excluded_by_source_label']=comparison.label.map(excluded.source_label.value_counts()).fillna(0).astype(int)
comparison['net_change']=comparison.after_retained-comparison.before_files
comparison['percent_after']=comparison.after_retained/max(len(cleaned),1)*100
save_table(comparison,'before_after_comparison.csv');display(comparison)
fig,axes=plt.subplots(1,2,figsize=(13,4))
comparison.set_index('label')[['before_files','after_retained']].plot.bar(ax=axes[0],rot=15)
axes[0].set(title='Jumlah sebelum dan sesudah',ylabel='Jumlah file/gambar')
share_before=comparison.before_files/max(len(before),1)*100
x=np.arange(len(CLASSES));axes[1].bar(x-.18,share_before,.36,label='Sebelum')
axes[1].bar(x+.18,comparison.percent_after,.36,label='Sesudah')
axes[1].set_xticks(x,CLASSES,rotation=15);axes[1].set(title='Proporsi kelas',ylabel='Persen');axes[1].legend()
save_figure('comparison_01_class_counts_shares.png')
reason_counts=excluded.groupby(['source_label','cleaning_reason'],dropna=False).size().rename('count').reset_index()
save_table(reason_counts,'exclusion_reasons_by_class.csv');display(reason_counts)
if len(excluded):
    reasons=excluded.cleaning_reason.value_counts()
    fig,ax=plt.subplots(figsize=(11,max(3,.45*len(reasons))))
    ax.barh(reasons.index.astype(str),reasons.values,color='#ef4444');ax.invert_yaxis()
    ax.set(title='Alasan eksklusi/karantina',xlabel='Jumlah file');save_figure('comparison_02_exclusion_reasons.png')
transitions=pd.crosstab(audit.source_label,audit.label).reindex(index=CLASSES,columns=CLASSES,fill_value=0)
save_table(transitions.rename_axis('source_label').reset_index(),'label_transition_table.csv')
print('Transisi label semua file, bukan confusion matrix prediksi:');display(transitions)
rows=[]
for stage,frame in [('before',before),('after',cleaned)]:
    counts=frame.label.value_counts().reindex(CLASSES,fill_value=0)
    ratio=float(counts.max()/counts.min()) if len(counts) and counts.min()>0 else None
    rows.append({'stage':stage,'n_files':len(frame),'largest_smallest_class_ratio':ratio,
        'readable':int(frame.readability_status.eq('readable').sum()),'below_minimum_flagged':int(frame.below_minimum.sum()),
        'median_width':pd.to_numeric(frame.width,errors='coerce').median(),
        'median_height':pd.to_numeric(frame.height,errors='coerce').median()})
summary_comparison=pd.DataFrame(rows);save_table(summary_comparison,'before_after_summary.csv');display(summary_comparison)
metric_comparison=[]
for stage,frame in [('before',metrics),('after',after_metrics)]:
    sample=frame[frame.metrics_status.eq('ok')]
    for col in QUALITY_COLS:
        v=pd.to_numeric(sample[col],errors='coerce').dropna()
        metric_comparison.append({'stage':stage,'metric':col,'sample_n':len(v),'sample_median':v.median() if len(v) else None})
save_table(pd.DataFrame(metric_comparison),'visual_sample_before_after_medians.csv')


## 12. Manifest, kamus data, laporan dan backup final
`manifest_after_technical_cleaning.csv` hanya kandidat untuk tahap audit selanjutnya. `include` tetap nullable; jangan fillna(1). `sha256` sengaja kosong dengan hash_status=deferred. `group_id` kosong, bukan ID buatan.

Jika ingin review manual: unduh/copy manual_review_template.csv, isi baris yang ditinjau, simpan dengan nama berbeda, lalu tetapkan MANUAL_REVIEW_CSV dan jalankan ulang bagian 9–12. Template asli dan manifest sebelum cleaning tetap menjadi baseline.


In [ ]:
dictionary={
 'image_id':'ID urut path dalam RUN ini; bukan hash/patient ID; cocokkan relative_path saat lintas run',
 'relative_path':'Path relatif terhadap DATA_ROOT; sumber tidak diubah',
 'label':'Label folder sebelum review; bisa berubah hanya dengan label_override dan bukti',
 'source_label':'Label folder asli; tetap dicatat setelah review',
 'width':'Lebar header sebelum orientasi EXIF (piksel)',
 'height':'Tinggi header sebelum orientasi EXIF (piksel)',
 'display_width':'Lebar setelah memperhitungkan orientasi EXIF',
 'display_height':'Tinggi setelah memperhitungkan orientasi EXIF',
 'format':'Format isi Pillow, bukan ekstensi saja',
 'mode':'Mode warna sumber: RGB/L/RGBA/CMYK/P dan lainnya',
 'readability_status':'Hasil pemeriksaan header/load/RGB atau alasan skip',
 'readable':'True decode berhasil; False decode gagal; kosong jika tidak didekode',
 'rgb_convertible':'Konversi RGB berhasil saat inspeksi; bukan file sumber diubah',
 'sha256':'Kosong; audit hash ditunda, jangan mengklaim bebas duplikat',
 'hash_status':'deferred di seluruh notebook ini',
 'original_status':'unknown di seluruh notebook; token filename bukan bukti',
 'group_id':'Kosong sampai audit lineage; jangan isi image_id untuk memaksakan grouped split',
 'include':'Kosong=pending audit; 0=eksklusi/karantina; 1=keep manual dengan alasan; tidak menjamin siap split',
 'technical_keep':'Lolos kebijakan teknis citra statis dan ukuran; bukan review label/lineage',
 'retained_for_eda':'technical_keep dan tidak dieksklusi manual; dipakai EDA sesudah cleaning',
 'ready_for_split':'Selalu False; hash, lineage dan duplikat belum diaudit',
 'audit_note':'Alasan teknis, kebijakan dan keputusan manual yang dicatat',
 'cleaning_reason':'Alasan eksklusi atau karantina; tidak menghapus sumber',
 'filename_aug_hint':'Petunjuk regex nama file, tidak mengubah original_status',
 'below_minimum':'Salah satu sisi < MIN_SIDE; flag saja kecuali policy exclude',
 'over_maximum':'Di atas MAX_PIXELS atau ditolak bomb limit; tidak sama dengan korup',
 'extension_format_mismatch':'Ekstensi berbeda dari format isi; jika keduanya diizinkan hanya review flag',
 'has_exif':'Indikasi adanya EXIF; metadata sensitif tidak disimpan',
 'has_alpha':'Indikasi alpha/transparansi; komposit putih hanya untuk statistik/preview',
 'n_frames':'Jumlah frame; frame0 diperiksa, multiframe dikarantina dari kandidat statis',
 'file_size_bytes':'Ukuran berkas, bukan ukuran piksel',
 'megapixels':'width*height/1e6 dari header',
 'aspect_ratio':'width/height sebelum orientasi',
 'manual_decision':'keep/exclude dari review yang tervalidasi',
 'manual_reason':'Alasan manual wajib untuk keputusan aktif',
 'reviewer':'Peninjau yang dicatat; wajib untuk keputusan aktif',
 'label_evidence':'Bukti sumber/review untuk perubahan label',
 'brightness_mean':'Rata-rata grayscale thumbnail; bukan tipe warna kulit',
 'contrast_std':'Standar deviasi grayscale thumbnail',
 'saturation_mean':'Rata-rata saturasi HSV, 0–1',
 'entropy':'Entropy histogram grayscale thumbnail; bit',
 'laplacian_variance':'Varians Laplacian 4-neighbor thumbnail <=256px; bukan ambang blur universal',
 'hist_channel_bin':'Histogram RGB 32 bin per channel, dinormalisasi per gambar; label hist_0_00..hist_2_31'
}
for col in audit.columns:dictionary.setdefault(col,'Metadata pendukung; lihat fungsi inspeksi atau cleaning di notebook.')
save_table(pd.DataFrame([{'column':c,'definition':d} for c,d in dictionary.items()]),'data_dictionary.csv')
config={'run_id':RUN_ID,'actual_data_root':str(DATA_ROOT),'requested_data_source':str(DATA_SOURCE),
 'local_content_verified_against_drive':False,'seed':SEED,'max_pixels':MAX_PIXELS,
 'min_side':MIN_SIDE,'minimum_policy':MINIMUM_POLICY,'minimum_reason':MINIMUM_POLICY_REASON,
 'workers':MAX_WORKERS,'visual_sample_per_class':VISUAL_SAMPLE_PER_CLASS,
 'manual_review_csv':str(MANUAL_REVIEW_CSV),'scan_seconds':scan_seconds,
 'hash_performed':False,'duplicates_audited':False,'lineage_audited':False,'split_created':False}
environment={'python':sys.version,'platform':platform.platform(),'timestamp_utc':datetime.now(timezone.utc).isoformat(),
 'packages':{p:metadata.version(p) for p in ['numpy','pandas','Pillow','matplotlib']},'gpu_required':False}
(WORK_DIR/'config.json').write_text(json.dumps(config,indent=2))
(WORK_DIR/'environment.json').write_text(json.dumps(environment,indent=2))
comparison_lines='\n'.join(f'| {r.label} | {r.before_files} | {r.after_retained} | {r.excluded_by_source_label} |' for r in comparison.itertuples())
manual_rows=int(audit.manual_decision.ne('').sum())
report=f"""# Laporan EDA dan Cleaning Teknis KulitKu

Run {RUN_ID}; sumber aktual {DATA_ROOT}; seed {SEED}.

## Cakupan
EDA metadata seluruh file, pemeriksaan decode citra statis, cleaning teknis dan review manual yang tersedia, lalu EDA sesudah. Hash/duplikat/lineage ditunda. Tidak ada split/training.

## Perbandingan
| Kelas | File awal | Kandidat sesudah | Eksklusi menurut label sumber |
|---|---:|---:|---:|
{comparison_lines}

Total awal {len(before)}, dipertahankan {len(cleaned)}, dikeluarkan/karantina {len(excluded)}. Perubahan label dapat memindahkan hitungan kelas; lihat label_transition_table.csv.

## Kebijakan
Batas maksimum {MAX_PIXELS/1e6:g}MP. Minimum per sisi {MIN_SIDE}px dengan kebijakan {MINIMUM_POLICY}. Alasan minimum: {MINIMUM_POLICY_REASON or 'belum dijadikan aturan eksklusi'}. RGB-convertible dipertahankan. Multiframe ditahan karena tujuan citra statis. Tidak ada gambar dihapus dari sumber.

## Review manual
Baris dengan keputusan manual: {manual_rows}. {'Review semantik belum dilakukan melalui CSV; cleaning ini hanya teknis.' if manual_rows==0 else 'Keputusan berlaku hanya untuk baris yang dicatat; bukan klaim seluruh dataset terverifikasi.'}

## Makna EDA visual
Statistik visual hanya sampel hingga {VISUAL_SAMPLE_PER_CLASS} gambar per kelas. Sesudah cleaning memakai subset dari sampel awal, dengan label hasil review jika ada; ukuran sampel bisa berkurang. Lihat before_visual_sample_counts.csv dan after_visual_sample_counts.csv. Grafik resolusi meliputi file yang memiliki metadata, tidak mencakup header yang gagal dibaca. RGB/brightness tidak menentukan warna kulit pasien; histogram tidak menjamin diagnosis. Contoh watermark/latar/label harus diperiksa manusia.

## Batas kesimpulan
Status original unknown dan group_id kosong. Tidak ada patient ID atau test eksternal. File byte/piksel identik belum dicari, augmentasi belum dikelompokkan. Keseimbangan gambar bukan keseimbangan pasien independen. Tidak boleh langsung split/training menggunakan manifest ini. Dataset sumber, lisensi dan anotasi perlu didokumentasikan sesuai protokol penelitian.

## Output dan tindak lanjut
manifest_before_cleaning.csv, manifest_cleaning_decisions.csv, manifest_after_technical_cleaning.csv, excluded_or_quarantined.csv, manual_review_template.csv, before_after_comparison.csv, visual_sample_metrics.csv dan figures/. Isi review yang diperlukan, selesaikan audit duplikat dan lineage, baru rancang split berdasarkan kelompok.
"""
(WORK_DIR/'EDA_CLEANING_REPORT.md').write_text(report)
assert len(before)==len(cleaned)+len(excluded)
assert audit.original_status.eq('unknown').all() and audit.group_id.eq('').all()
assert audit.sha256.eq('').all() and not audit.ready_for_split.any()
assert not cleaned.image_id.isin(excluded.image_id).any()
# CSV roundtrip preserves pending include, rather than filling it with 1.
roundtrip=pd.read_csv(WORK_DIR/'manifest_cleaning_decisions.csv',dtype={'include':'Int64'})
assert roundtrip['include'].isna().sum()==audit['include'].isna().sum()
for path in tqdm(sorted(WORK_DIR.rglob('*')),desc='Backup hasil'):
    if path.is_file():atomic_copy(path,RUN_DIR/path.relative_to(WORK_DIR))
print('SELESAI sampai EDA setelah cleaning.')
print('Hasil:',RUN_DIR)
print('Laporan:',RUN_DIR/'EDA_CLEANING_REPORT.md')
print('Manifest kandidat:',RUN_DIR/'manifest_after_technical_cleaning.csv')
print('Dataset sumber tetap utuh. Audit duplikat/augmentasi masih menunggu sebelum split.')


## Selesai — berhenti di sini
Simpan notebook di `machine-learning/notebooks/01_eda_cleaning.ipynb`. Seluruh output berada di Drive pada run terpisah. Dataset yang lolos cleaning teknis tidak otomatis menjadi dataset final penelitian.

Bila sel selesai tanpa perubahan jumlah signifikan, itu tetap hasil yang sah: banyak file mungkin sudah terbaca dan memenuhi aturan teknis. Jangan mencari aturan tambahan hanya untuk membuang data atau menaikkan skor.

**Referensi implementasi:** [Pillow Image](https://pillow.readthedocs.io/en/stable/reference/Image.html), [ImageOps EXIF transpose](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html), [pandas nullable integers](https://pandas.pydata.org/docs/user_guide/integer_na.html). Batas teknis adalah keputusan penelitian, bukan ambang klinis dari dokumentasi tersebut.
